
# AMML Ranking Competition 2026

## Ранжирование текстовых документов с использованием Jina Reranker v3.5 и LoRA

Цель работы – построение модели ранжирования документов по степени их релевантности поисковому запросу. Качество ранжирования оценивается метрикой NDCG@10.

В качестве основного подхода используется предобученная модель Jina Reranker v3.5 с последующим дообучением методом Low-Rank Adaptation (LoRA). Для оптимизации применяется функция потерь спискового ранжирования (Listwise Ranking Loss).

Итоговое решение включает следующие этапы:

1. Загрузка и предобработка данных.
2. Формирование обучающей и валидационной выборок.
3. Отбор сложных отрицательных примеров (Hard Negative Mining).
4. Дообучение модели Jina Reranker v3.5 с использованием LoRA.
5. Получение предсказаний для тестовой выборки.
6. Формирование итогового файла submission.csv.

Обучение исходного решения проводилось в среде Linux (Kaggle) с использованием NVIDIA Tesla T4. Воспроизведение пайплайна выполняется в Google Colab на GPU того же типа.

Результат исходного решения на Public Leaderboard: NDCG@10 = 0.966072.

Описание промежуточных экспериментов и сравнение подходов представлены в файле README.md.

(☉_☉)

𓀡

𓀥𓀤𓀣

In [ ]:

import sys
import platform
import random
import gc
import re
import importlib.metadata as metadata
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import GroupShuffleSplit
from tqdm.auto import tqdm

SEED = 2026

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

DATA_DIR = Path("/content")
OUTPUT_DIR = DATA_DIR / "results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Операционная система:", platform.system())
print("Версия Python:", sys.version.split()[0])
print("Версия PyTorch:", torch.__version__)
print("Устройство:", DEVICE)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

packages = [
    "numpy",
    "pandas",
    "torch",
    "transformers",
    "peft",
    "accelerate",
    "safetensors",
    "scikit-learn",
    "torchao",
    "einops"
]

for package in packages:
    try:
        print(f"{package}: {metadata.version(package)}")
    except metadata.PackageNotFoundError:
        print(f"{package}: не установлен")


Операционная система: Linux
Версия Python: 3.13.16
Версия PyTorch: 2.14.1+cu130
Устройство: cuda
GPU: Tesla T4
numpy: 2.4.6
pandas: 2.2.3
torch: 2.14.1+cu130
transformers: 5.22.1
peft: 0.21.2
accelerate: 1.15.0
safetensors: 0.8.0
scikit-learn: 1.6.1
torchao: 0.10.0
einops: 0.8.2



## 1. Загрузка данных

В соревновании используются две выборки: обучающая с известными значениями релевантности документов и тестовая, для которой необходимо сформировать предсказания.

Каждому поисковому запросу соответствует группа документов. Признак `relevance` содержит целевую переменную, а `query_id` позволяет объединять документы одного запроса.

Разделение на обучающую и валидационную выборки выполняется по идентификаторам запросов, чтобы документы одного запроса не попадали одновременно в разные части данных.


In [ ]:

train = pd.read_csv(
    DATA_DIR / "train.csv",
    dtype={"id": str, "query_id": str}
)

test = pd.read_csv(
    DATA_DIR / "test.csv",
    dtype={"id": str, "query_id": str}
)

sample = pd.read_csv(
    DATA_DIR / "sample_submission.csv",
    dtype={"id": str}
)

assert train["id"].is_unique
assert test["id"].is_unique
assert sample["id"].tolist() == test["id"].tolist()

print("Обучающая выборка:", train.shape)
print("Тестовая выборка:", test.shape)
print("Количество обучающих запросов:", train["query_id"].nunique())
print("Количество тестовых запросов:", test["query_id"].nunique())

print("\nРаспределение релевантности:")
print(train["relevance"].value_counts().sort_index())

print("\nПропущенные значения:")
print(train.isna().sum())

display(train.head(3))


Обучающая выборка: (31530, 12)
Тестовая выборка: (13507, 11)
Количество обучающих запросов: 3042
Количество тестовых запросов: 1305

Распределение релевантности:
relevance
0    23362
1     8168
Name: count, dtype: int64

Пропущенные значения:
id                          0
query_id                    0
document_id                 0
article_id                  0
query                       0
title                    1405
text                        0
passage_index            1398
article_passage_count    1959
article_char_count       2534
ingest_source               0
relevance                   0
dtype: int64


,id,query_id,document_id,article_id,query,title,text,passage_index,article_passage_count,article_char_count,ingest_source,relevance
0,r_0000a01ae1fc85409cfc,q_0c47372a2c70b5d45a3e,d_3d8b8c509b5daa75f1a3,a_9cb99738b80787ceb3b0,Сколько частей у фильма Властелин Колец?,Влияние и адаптации произведений Толкина,В 1968 году писатель продал права на экранизац...,4.0,72.0,27472,S0,0
1,r_0000d01ccdb12a83c85e,q_f692ddf7b1cf937cfda9,d_8a4095cfd564a399f67c,a_f0741cd81ca38fbb51c7,Где находится Кременчуг?,Кременчуг,Город является крупным железнодорожным узлом Ц...,38.0,47.0,9783,S3,1
2,r_000385ed0a8e1ebfe0e5,q_06c523053ba77923e44c,d_234baccfeddd33d157ca,a_77984ff4169b47f83448,Были ли краёвцы в России?,Краёвцы,Среди краёвцев выделилась два направления (либ...,17.0,74.0,69424,S0,0



## 2. Предобработка данных

Для каждого документа формируется единое текстовое представление путём объединения заголовка (`title`) и основного текста (`text`). Максимальная длина полученной строки ограничивается 1800 символами.

При анализе тестовой выборки были обнаружены ошибки кодировки в отдельных документах. Для их исправления применяется восстановление текста из кодировки Windows-1251 в UTF-8. Исправление выполняется отдельно для заголовка и основного текста и применяется только в случаях, когда количество признаков повреждённой кодировки уменьшается.

Обучающая выборка обрабатывается без исправления кодировки, чтобы сохранить исходные условия дообучения модели.


In [ ]:

train["document"] = (
    train["title"].fillna("").astype(str)
    + ". "
    + train["text"].fillna("").astype(str)
).str.slice(0, 1800)

mojibake = re.compile(
    r"(?:Р[°µёѕ»їґЅЎІ]|С[‚ЃЏЉЊ]|вЂ)"
)

def bad_count(text):
    return len(mojibake.findall(text))


def repair_text(value):
    if pd.isna(value):
        return ""

    original = str(value)

    if bad_count(original) < 3:
        return original

    try:
        candidate = original.encode("cp1251").decode("utf-8")

        if bad_count(candidate) < bad_count(original):
            return candidate

    except (UnicodeError, ValueError):
        pass

    parts = re.split(r"(\n+)", original)
    repaired = []

    for part in parts:
        if bad_count(part) < 3:
            repaired.append(part)
            continue

        try:
            candidate = part.encode("cp1251").decode("utf-8")

            if bad_count(candidate) < bad_count(part):
                repaired.append(candidate)
            else:
                repaired.append(part)

        except (UnicodeError, ValueError):
            repaired.append(part)

    return "".join(repaired)


test["title_clean"] = test["title"].map(repair_text)
test["text_clean"] = test["text"].map(repair_text)

original_documents = (
    test["title"].fillna("").astype(str)
    + ". "
    + test["text"].fillna("").astype(str)
).str.slice(0, 1800)

test["document"] = (
    test["title_clean"]
    + ". "
    + test["text_clean"]
).str.slice(0, 1800)

changed = original_documents != test["document"]

print("Документов в обучающей выборке:", len(train))
print("Документов в тестовой выборке:", len(test))
print("Исправлено документов:", int(changed.sum()))
print("Затронуто запросов:", test.loc[changed, "query_id"].nunique())


Документов в обучающей выборке: 31530
Документов в тестовой выборке: 13507
Исправлено документов: 72
Затронуто запросов: 70



## 3. Формирование обучающей и валидационной выборок

Для оценки качества модели обучающие данные разделяются на несколько непересекающихся групп поисковых запросов.

Разделение выполняется по `query_id`, поскольку одному запросу соответствует несколько документов. Это позволяет исключить попадание документов одного запроса одновременно в обучающую и валидационную выборки.

В итоговом эксперименте использовались:

- **Train** – 2408 запросов для дообучения модели.
- **Validation** – 329 запросов для оценки качества во время разработки.
- **Dev** – 305 запросов для дополнительной проверки обобщающей способности модели.

Все разделения выполняются с фиксированными значениями генератора случайных чисел для обеспечения воспроизводимости.

После формирования выборок проверяется отсутствие пересечений между группами запросов.


In [ ]:

split_initial = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

_, initial_valid_idx = next(
    split_initial.split(train, groups=train["query_id"])
)

initial_valid_qids = set(
    train.iloc[initial_valid_idx]["query_id"]
)

available_qids = sorted(
    set(train["query_id"]) - initial_valid_qids
)

rng_holdout = np.random.default_rng(2026)

holdout_qids = set(
    rng_holdout.choice(
        available_qids,
        size=400,
        replace=False
    )
)

teacher_qids = (
    set(train["query_id"])
    - initial_valid_qids
    - holdout_qids
)

teacher_rows = train[
    train["query_id"].isin(teacher_qids)
].copy()

split_teacher = GroupShuffleSplit(
    n_splits=1,
    test_size=0.15,
    random_state=2026
)

train_idx, dev_idx = next(
    split_teacher.split(
        teacher_rows,
        groups=teacher_rows["query_id"]
    )
)

dev_qids = set(
    teacher_rows.iloc[dev_idx]["query_id"]
)

old_train_qids = set(
    teacher_rows.iloc[train_idx]["query_id"]
)

old_train = train[
    train["query_id"].isin(old_train_qids)
].copy()

combined = (
    old_train["title"].fillna("").astype(str)
    + " "
    + old_train["text"].fillna("").astype(str)
)

bad_qids = set(
    old_train.loc[
        combined.str.count(mojibake) >= 3,
        "query_id"
    ]
)

clean_qids = np.array(
    sorted(old_train_qids - bad_qids)
)

rng_split = np.random.default_rng(4701)
rng_split.shuffle(clean_qids)

n_val = round(0.2 * len(clean_qids))
val_qids = set(clean_qids[:n_val])

fit_qids = (
    set(train["query_id"])
    - dev_qids
    - val_qids
)

fit = train[
    train["query_id"].isin(fit_qids)
].copy()

val = train[
    train["query_id"].isin(val_qids)
].copy()

dev = train[
    train["query_id"].isin(dev_qids)
].copy()

assert fit_qids.isdisjoint(val_qids)
assert fit_qids.isdisjoint(dev_qids)
assert val_qids.isdisjoint(dev_qids)

assert len(teacher_qids) == 2033
assert len(fit_qids) == 2408
assert len(val_qids) == 329
assert len(dev_qids) == 305
assert len(fit) == 24944

print("Запросов для расчёта оценок Jina:", len(teacher_qids))
print("Обучающих запросов:", len(fit_qids))
print("Обучающих документов:", len(fit))
print("Запросов Validation:", len(val_qids))
print("Документов Validation:", len(val))
print("Запросов Dev:", len(dev_qids))
print("Документов Dev:", len(dev))


Запросов для расчёта оценок Jina: 2033
Обучающих запросов: 2408
Обучающих документов: 24944
Запросов Validation: 329
Документов Validation: 3426
Запросов Dev: 305
Документов Dev: 3160



## 4. Инициализация предобученной модели

В качестве базовой модели используется `jinaai/jina-reranker-v3.5`, предназначенная для оценки релевантности пары «запрос – документ».

Модель загружается с платформы Hugging Face и переносится на GPU при его наличии. Для сокращения потребления видеопамяти используется точность FP16.

На следующих этапах модель применяется для получения оценок релевантности документов, после чего выполняется параметрически эффективное дообучение методом LoRA.

Основные параметры:
- Базовая модель – `jinaai/jina-reranker-v3.5`.
- Тип вычислений – FP16 на GPU.
- Максимальная длина документа – 192 токена.
- Метод дообучения – LoRA.


In [ ]:

from transformers import AutoModel, AutoTokenizer

MODEL_ID = "jinaai/jina-reranker-v3.5"
MAX_DOC_TOKENS = 192

MODEL_DTYPE = (
    torch.float16 if DEVICE.type == "cuda"
    else torch.float32
)

gc.collect()

if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True
)

tokenizer.padding_side = "left"

model = AutoModel.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    dtype=MODEL_DTYPE
)

model = model.to(DEVICE)
model.eval()

total_params = sum(
    p.numel() for p in model.parameters()
)

print("Модель:", MODEL_ID)
print("Устройство:", next(model.parameters()).device)
print("Тип данных:", next(model.parameters()).dtype)
print("Количество параметров:", total_params)
print("Модель успешно загружена")


Loading weights:   0%|          | 0/312 [00:00<?, ?it/s]

Модель: jinaai/jina-reranker-v3.5
Устройство: cuda:0
Тип данных: torch.float16
Количество параметров: 596836352
Модель успешно загружена



## 5. Расчёт оценок релевантности базовой модели

Перед дообучением необходимо получить исходные оценки релевантности документов с помощью предобученной Jina Reranker v3.5.

Для каждой пары «запрос – документ» модель рассчитывает числовую оценку (`score`), отражающую предполагаемую релевантность документа запросу.

Полученные оценки используются для отбора сложных отрицательных примеров (Hard Negative Mining). К таким примерам относятся нерелевантные документы, которым базовая модель присваивает относительно высокие оценки.

Использование сложных отрицательных примеров позволяет сосредоточить дообучение на документах, которые модель первоначально затрудняется правильно ранжировать.

Исходные значения `relevance` сохраняются как целевая переменная, а оценки базовой модели применяются только как вспомогательный инструмент формирования обучающих групп.


In [ ]:

def score_jina_group(group):
    query = str(group["query"].iloc[0])

    documents = []

    for document in group["document"].fillna("").astype(str):
        token_ids = tokenizer.encode(
            document,
            add_special_tokens=False,
            truncation=True,
            max_length=MAX_DOC_TOKENS
        )

        documents.append(
            tokenizer.decode(
                token_ids,
                skip_special_tokens=True
            )
        )

    with torch.no_grad():
        ranked = model.rerank(
            query,
            documents,
            top_n=len(documents)
        )

    scores = np.full(
        len(documents),
        np.nan,
        dtype=np.float32
    )

    for result in ranked:
        scores[int(result["index"])] = float(
            result["relevance_score"]
        )

    assert np.isfinite(scores).all()

    output = group[["id", "query_id"]].copy()
    output["jina_score"] = scores

    return output


# Проверка на трёх документах одного запроса
probe_qid = sorted(teacher_qids)[0]

probe_group = train[
    train["query_id"] == probe_qid
].head(3)

probe_scores = score_jina_group(probe_group)

probe_result = probe_scores.merge(
    probe_group[["id", "relevance"]],
    on="id",
    how="left"
)

print("Проверочный запрос:", probe_group["query"].iloc[0])
print("Оценено документов:", len(probe_scores))

display(
    probe_result[
        ["id", "relevance", "jina_score"]
    ].sort_values(
        "jina_score",
        ascending=False
    )
)


Проверочный запрос: Какая площадь города Мумбаи?
Оценено документов: 3


,id,relevance,jina_score
1,r_12a1d1e218b0bec36593,0,0.214586
2,r_291ba0088b42f624d892,0,-0.073537
0,r_0d9ee7b274341e1ea247,0,-0.130878


In [ ]:

SCORES_PATH = OUTPUT_DIR / "teacher_scores.csv"
SAVE_EVERY = 25

teacher_data = train[
    train["query_id"].isin(teacher_qids)
].copy()

expected_ids = set(teacher_data["id"])

if SCORES_PATH.exists():
    saved_scores = pd.read_csv(
        SCORES_PATH,
        dtype={"id": str, "query_id": str}
    )
    print("Найден сохранённый файл:", len(saved_scores), "строк")
else:
    saved_scores = pd.DataFrame(
        columns=["id", "query_id", "jina_score"]
    )

assert saved_scores["id"].is_unique
assert set(saved_scores["id"]).issubset(expected_ids)

completed_qids = set(saved_scores["query_id"])

pending = []

groups = teacher_data.groupby(
    "query_id",
    sort=True
)

for qid, group in tqdm(
    groups,
    total=len(teacher_qids),
    desc="Jina teacher scores"
):
    if qid in completed_qids:
        continue

    group_scores = score_jina_group(group)
    pending.append(group_scores)

    if len(pending) >= SAVE_EVERY:
        saved_scores = pd.concat(
            [saved_scores, *pending],
            ignore_index=True
        )

        saved_scores.to_csv(
            SCORES_PATH,
            index=False
        )

        pending = []

if pending:
    saved_scores = pd.concat(
        [saved_scores, *pending],
        ignore_index=True
    )

    saved_scores.to_csv(
        SCORES_PATH,
        index=False
    )

assert saved_scores["id"].is_unique
assert set(saved_scores["id"]) == expected_ids
assert np.isfinite(
    saved_scores["jina_score"].astype(float)
).all()

print("\nРасчёт завершён")
print("Обработано запросов:", saved_scores["query_id"].nunique())
print("Оценено документов:", len(saved_scores))
print("Файл сохранён:", SCORES_PATH)

display(saved_scores.head())


Jina teacher scores:   0%|          | 0/2033 [00:00<?, ?it/s]

/tmp/ipykernel_1367/2212679068.py:45: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  saved_scores = pd.concat(



Расчёт завершён
Обработано запросов: 2033
Оценено документов: 21052
Файл сохранён: /content/results/teacher_scores.csv


,id,query_id,jina_score
0,r_0d9ee7b274341e1ea247,q_00296e2e8b6b71f9768d,-0.112327
1,r_12a1d1e218b0bec36593,q_00296e2e8b6b71f9768d,0.092147
2,r_291ba0088b42f624d892,q_00296e2e8b6b71f9768d,-0.088006
3,r_3681fc839d2717be33eb,q_00296e2e8b6b71f9768d,-0.059072
4,r_8480e48e78827e1025b2,q_00296e2e8b6b71f9768d,0.270397


In [ ]:

REVERSE_PATH = OUTPUT_DIR / "teacher_scores_reverse.csv"
COMBINED_PATH = OUTPUT_DIR / "teacher_scores_combined.csv"
SAVE_EVERY = 25

teacher_data = train[
    train["query_id"].isin(teacher_qids)
].copy()

forward_scores = pd.read_csv(
    SCORES_PATH,
    dtype={"id": str, "query_id": str}
)

assert forward_scores["id"].is_unique
assert set(forward_scores["id"]) == set(teacher_data["id"])

if REVERSE_PATH.exists():
    reverse_scores = pd.read_csv(
        REVERSE_PATH,
        dtype={"id": str, "query_id": str}
    )
else:
    reverse_scores = pd.DataFrame(
        columns=["id", "query_id", "reverse_score"]
    )

assert reverse_scores["id"].is_unique
assert set(reverse_scores["id"]).issubset(
    set(teacher_data["id"])
)

expected_counts = teacher_data.groupby("query_id").size()
saved_counts = reverse_scores.groupby("query_id").size()

completed_qids = {
    qid for qid, count in saved_counts.items()
    if count == expected_counts.loc[qid]
}

reverse_scores = reverse_scores[
    reverse_scores["query_id"].isin(completed_qids)
].copy()

pending = []

for qid, group in tqdm(
    teacher_data.groupby("query_id", sort=True),
    total=len(teacher_qids),
    desc="Jina reverse scores"
):
    if qid in completed_qids:
        continue

    result = score_jina_group(
        group.iloc[::-1]
    ).rename(
        columns={"jina_score": "reverse_score"}
    )

    pending.append(result)

    if len(pending) >= SAVE_EVERY:
        batch = pd.concat(pending, ignore_index=True)

        reverse_scores = (
            pd.concat(
                [reverse_scores, batch],
                ignore_index=True
            )
            if not reverse_scores.empty
            else batch
        )

        temp_path = REVERSE_PATH.with_suffix(".tmp")
        reverse_scores.to_csv(temp_path, index=False)
        temp_path.replace(REVERSE_PATH)

        pending = []

if pending:
    batch = pd.concat(pending, ignore_index=True)

    reverse_scores = (
        pd.concat(
            [reverse_scores, batch],
            ignore_index=True
        )
        if not reverse_scores.empty
        else batch
    )

    reverse_scores.to_csv(REVERSE_PATH, index=False)

assert reverse_scores["id"].is_unique
assert set(reverse_scores["id"]) == set(teacher_data["id"])

combined = forward_scores.merge(
    reverse_scores,
    on=["id", "query_id"],
    validate="one_to_one"
)

def group_zscore(series):
    values = series.to_numpy(dtype=np.float64)
    std = values.std(ddof=1)

    if not np.isfinite(std) or std < 1e-12:
        return pd.Series(
            np.zeros(len(series)),
            index=series.index
        )

    return pd.Series(
        (values - values.mean()) / (std + 1e-6),
        index=series.index
    )

combined["forward_z"] = combined.groupby(
    "query_id"
)["jina_score"].transform(group_zscore)

combined["reverse_z"] = combined.groupby(
    "query_id"
)["reverse_score"].transform(group_zscore)

combined["teacher_average"] = (
    combined["forward_z"] + combined["reverse_z"]
) / 2

assert len(combined) == len(teacher_data)
assert np.isfinite(combined["teacher_average"]).all()

combined.to_csv(COMBINED_PATH, index=False)

print("Обработано запросов:", combined["query_id"].nunique())
print("Оценено документов:", len(combined))
print("Файл сохранён:", COMBINED_PATH)

display(
    combined[
        ["id", "query_id", "jina_score",
         "reverse_score", "teacher_average"]
    ].head()
)


Jina reverse scores:   0%|          | 0/2033 [00:00<?, ?it/s]

Обработано запросов: 2033
Оценено документов: 21052
Файл сохранён: /content/results/teacher_scores_combined.csv


,id,query_id,jina_score,reverse_score,teacher_average
0,r_0d9ee7b274341e1ea247,q_00296e2e8b6b71f9768d,-0.112327,-0.131066,-1.044370
1,r_12a1d1e218b0bec36593,q_00296e2e8b6b71f9768d,0.092147,0.020546,-0.357555
2,r_291ba0088b42f624d892,q_00296e2e8b6b71f9768d,-0.088006,-0.075329,-0.888409
3,r_3681fc839d2717be33eb,q_00296e2e8b6b71f9768d,-0.059072,-0.078725,-0.840172
4,r_8480e48e78827e1025b2,q_00296e2e8b6b71f9768d,0.270397,0.060423,0.059383



## 6. Формирование обучающих групп (Hard Negative Mining)

Для дообучения модели используется метод Hard Negative Mining – отбор сложных отрицательных примеров на основе оценок предобученной Jina Reranker v3.5.

Для каждого поискового запроса формируется группа из четырёх документов (`DOCS_PER_QUERY = 4`).

Алгоритм формирования группы:

1. Случайным образом выбираются до двух релевантных документов (`relevance = 1`).
2. Среди нерелевантных документов (`relevance = 0`) выбирается сложный отрицательный пример с максимальным значением `teacher_average`.
3. Оставшиеся позиции заполняются случайно выбранными документами с учётом уже отобранных примеров.
4. Порядок документов внутри группы случайным образом перемешивается.

Если для запроса отсутствуют оценки базовой модели, отрицательные примеры выбираются случайным образом.

Для обеспечения воспроизводимости используется фиксированное значение генератора случайных чисел.

Сформированные группы используются при дообучении модели с функцией потерь Listwise Ranking Loss, которая оптимизирует относительный порядок документов внутри каждого запроса.


In [ ]:

DOCS_PER_QUERY = 4
rng_mining = np.random.default_rng(SEED)

teacher_scores = pd.read_csv(
    COMBINED_PATH,
    dtype={"id": str, "query_id": str}
)

assert teacher_scores["id"].is_unique

fit_scored = fit.merge(
    teacher_scores[
        ["id", "query_id", "teacher_average"]
    ],
    on=["id", "query_id"],
    how="left",
    validate="one_to_one"
)


def mine_query_group(group, rng):
    selected = {}

    # 1. Выбираем до двух положительных документов
    positives = group.index[
        group["relevance"] > 0
    ].to_numpy()

    n_positive = min(2, len(positives))

    if n_positive:
        positive_idx = rng.choice(
            positives,
            size=n_positive,
            replace=False
        )

        for idx in positive_idx:
            selected[int(idx)] = "positive"

    # 2. Выбираем сложный отрицательный документ
    hard_candidates = group[
        (group["relevance"] == 0)
        & group["teacher_average"].notna()
    ]

    if not hard_candidates.empty:
        hard_idx = int(
            hard_candidates["teacher_average"].idxmax()
        )
        selected[hard_idx] = "hard_negative"

    else:
        negatives = group.index[
            group["relevance"] == 0
        ].to_numpy()

        if len(negatives):
            neg_idx = int(rng.choice(negatives))
            selected[neg_idx] = "random_negative"

    # 3. Дополняем группу уникальными документами
    remaining = group.index[
        ~group.index.isin(selected.keys())
    ].to_numpy()

    n_remaining = DOCS_PER_QUERY - len(selected)
    n_unique = min(n_remaining, len(remaining))

    unique_extra = rng.choice(
        remaining,
        size=n_unique,
        replace=False
    )

    # 4. При нехватке документов допускаем повторы
    n_padding = n_remaining - n_unique

    padded_extra = rng.choice(
        group.index.to_numpy(),
        size=n_padding,
        replace=True
    )

    chosen = (
        [(idx, typ) for idx, typ in selected.items()]
        + [(int(idx), "random_fill") for idx in unique_extra]
        + [(int(idx), "duplicate_padding") for idx in padded_extra]
    )

    # 5. Перемешиваем документы
    rng.shuffle(chosen)

    chosen_idx = [idx for idx, _ in chosen]

    result = group.loc[
        chosen_idx,
        [
            "id", "query_id", "query",
            "document", "relevance",
            "teacher_average"
        ]
    ].copy()

    result["selection_type"] = [
        typ for _, typ in chosen
    ]

    result["group_position"] = np.arange(
        DOCS_PER_QUERY
    )

    return result


# Формируем обучающие группы
train_groups = []

for qid, group in tqdm(
    fit_scored.groupby("query_id", sort=True),
    total=len(fit_qids),
    desc="Hard Negative Mining"
):
    train_groups.append(
        mine_query_group(group, rng_mining)
    )

mined_train = pd.concat(
    train_groups,
    ignore_index=True
)

# Проверки
group_sizes = mined_train.groupby("query_id").size()

assert group_sizes.eq(DOCS_PER_QUERY).all()
assert mined_train["query_id"].nunique() == len(fit_qids)
assert len(mined_train) == len(fit_qids) * DOCS_PER_QUERY
assert set(mined_train["query_id"]).isdisjoint(val_qids)
assert set(mined_train["query_id"]).isdisjoint(dev_qids)

MINED_PATH = OUTPUT_DIR / "mined_train_groups.csv"
mined_train.to_csv(MINED_PATH, index=False)

original_sizes = fit.groupby("query_id").size()

print("\nHard Negative Mining завершён")
print("Обучающих запросов:", mined_train["query_id"].nunique())
print("Документов:", len(mined_train))
print("Запросов с менее чем 4 документами:",
      int(original_sizes.lt(4).sum()))

print("\nТипы документов:")
print(mined_train["selection_type"].value_counts())

print("\nРаспределение релевантности:")
print(mined_train["relevance"].value_counts().sort_index())

print("\nЗапросов без положительных примеров:",
      int(mined_train.groupby("query_id")["relevance"]
          .max().eq(0).sum()))

print("\nФайл сохранён:", MINED_PATH)

display(mined_train.head(8))


Hard Negative Mining:   0%|          | 0/2408 [00:00<?, ?it/s]


Hard Negative Mining завершён
Обучающих запросов: 2408
Документов: 9632
Запросов с менее чем 4 документами: 21

Типы документов:
selection_type
positive             3972
random_fill          3212
hard_negative        1399
random_negative      1009
duplicate_padding      40
Name: count, dtype: int64

Распределение релевантности:
relevance
0    5317
1    4315
Name: count, dtype: int64

Запросов без положительных примеров: 0

Файл сохранён: /content/results/mined_train_groups.csv


,id,query_id,query,document,relevance,teacher_average,selection_type,group_position
0,r_b5c61745207ac64d4964,q_0013d76bda598608f643,Иоа́нн Тобо́льский был поэтом?,Иоанн Тобольский. Иоанн Максимович был самым п...,1,NaN,positive,0
1,r_6986c9ce8924aa47363b,q_0013d76bda598608f643,Иоа́нн Тобо́льский был поэтом?,"Иоанн Тобольский. Особо Иоанн поддерживал, в т...",0,NaN,random_fill,1
2,r_5b8619602bc083089f02,q_0013d76bda598608f643,Иоа́нн Тобо́льский был поэтом?,Иоанн Тобольский. Иоа́нн Тобо́льский (в миру И...,1,NaN,positive,2
3,r_2a9132d50090da1fccbf,q_0013d76bda598608f643,Иоа́нн Тобо́льский был поэтом?,"Толстой, Алексей Николаевич. Член Комиссии по ...",0,NaN,random_negative,3
4,r_988f84aa535040cf02ea,q_00296e2e8b6b71f9768d,Какая площадь города Мумбаи?,. Мумбаи — : в понятие города включают т. н. ...,1,0.980063,positive,0
5,r_d9fe7e0579ed0ddfbed2,q_00296e2e8b6b71f9768d,Какая площадь города Мумбаи?,Экономика Мумбаи. Столичный регион Мумбаи () о...,0,0.574563,hard_negative,1
6,r_291ba0088b42f624d892,q_00296e2e8b6b71f9768d,Какая площадь города Мумбаи?,Нави-Мумбаи. Строительство города-спутника Мум...,0,-0.888409,random_fill,2
7,r_8b8489298700e40c10a6,q_00296e2e8b6b71f9768d,Какая площадь города Мумбаи?,"Мумбаи. Мумбаи расположен в устье реки Улхас, ...",1,2.086932,positive,3



## 7. Дообучение модели методом LoRA

Для адаптации Jina Reranker v3.5 к задаче ранжирования используется Low-Rank Adaptation (LoRA).

Метод позволяет обучать небольшое количество дополнительных параметров, сохраняя основные веса предобученной модели замороженными. Это существенно снижает требования к видеопамяти GPU.

### Конфигурация обучения

- Базовая модель: `jinaai/jina-reranker-v3.5`.
- Метод адаптации: LoRA.
- Rank (`r`): 8.
- Alpha: 16.
- Dropout: 0.0.
- Целевые модули: `q_proj`, `v_proj`.
- Learning rate: `2e-5`.
- Оптимизатор: AdamW.
- Weight decay: 0.01.
- Количество эпох: 1.
- Размер группы: 4 документа на запрос.
- Функция потерь: Listwise Ranking Loss.
- Score scale: 8.0.
- Максимальная длина документа: 192 токена.
- Gradient clipping: 1.0.
- Точность вычислений: FP16.
- Gradient checkpointing: включён.
- Random seed: 2026.

Дообучение выполняется на сформированных группах документов. Модель учится присваивать релевантным документам более высокие оценки по сравнению с нерелевантными.

Для оптимизации используется списковая функция потерь, учитывающая относительный порядок документов внутри одного запроса.

Повторные позиции, возникшие при дополнении коротких групп, требуют отдельного учёта при вычислении функции потерь.

После обучения сохраняются веса LoRA-адаптера, которые могут быть повторно загружены без полного дообучения базовой модели.


In [ ]:

from peft import LoraConfig, PeftModel, get_peft_model

# Конфигурация LoRA
LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.0
LORA_TARGET_MODULES = ["q_proj", "v_proj"]

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGET_MODULES,
    bias="none",
    inference_mode=False
)

# Подключаем адаптер только один раз
if not isinstance(model, PeftModel):
    model = get_peft_model(
        model,
        lora_config
    )
else:
    print("LoRA уже подключена")

# Проверяем параметры модели
total_params = sum(
    p.numel() for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

lora_modules = [
    name
    for name, module in model.named_modules()
    if hasattr(module, "lora_A")
]

trainable_percent = (
    100 * trainable_params / total_params
)

print("Базовая модель:", MODEL_ID)
print("LoRA rank:", LORA_R)
print("LoRA alpha:", LORA_ALPHA)
print("LoRA dropout:", LORA_DROPOUT)
print("Target modules:", LORA_TARGET_MODULES)

print("\nВсего параметров:", total_params)
print("Обучаемых параметров:", trainable_params)
print("Доля обучаемых: {:.4f}%".format(
    trainable_percent
))
print("LoRA-модулей:", len(lora_modules))

assert trainable_params == 1_146_880, (
    f"Неожиданное количество параметров: {trainable_params}"
)

assert len(lora_modules) == 56, (
    f"Неожиданное количество LoRA-модулей: "
    f"{len(lora_modules)}"
)

print("\nLoRA успешно подключена!")


Базовая модель: jinaai/jina-reranker-v3.5
LoRA rank: 8
LoRA alpha: 16
LoRA dropout: 0.0
Target modules: ['q_proj', 'v_proj']

Всего параметров: 597983232
Обучаемых параметров: 1146880
Доля обучаемых: 0.1918%
LoRA-модулей: 56

LoRA успешно подключена!


In [ ]:

# Удаляем искусственно добавленные повторы
training_data = mined_train[
    mined_train["selection_type"] != "duplicate_padding"
].copy()

training_data = training_data.sort_values(
    ["query_id", "group_position"],
    kind="stable"
).reset_index(drop=True)

# Проверяем отсутствие повторяющихся документов
assert not training_data.duplicated(
    subset=["query_id", "id"]
).any()

# Проверяем метки релевантности
assert training_data["relevance"].isin([0, 1]).all()

# Формируем список групп для обучения
training_groups = [
    group.reset_index(drop=True)
    for _, group in training_data.groupby(
        "query_id",
        sort=True
    )
]

# Проверяем структуру групп
group_sizes = training_data.groupby("query_id").size()

assert len(training_groups) == 2408
assert group_sizes.le(DOCS_PER_QUERY).all()

positive_counts = training_data.groupby(
    "query_id"
)["relevance"].sum()

negative_counts = (
    group_sizes - positive_counts
)

print("Подготовка завершена")
print("Обучающих запросов:", len(training_groups))
print("Всего документов:", len(training_data))

print("\nКоличество документов в группах:")
print(group_sizes.value_counts().sort_index())

print("\nЗапросов без положительных примеров:",
      int(positive_counts.eq(0).sum()))

print("Запросов без отрицательных примеров:",
      int(negative_counts.eq(0).sum()))

print("\nПример обучающей группы:")
display(
    training_groups[0][
        ["id", "relevance", "selection_type"]
    ]
)


Подготовка завершена
Обучающих запросов: 2408
Всего документов: 9592

Количество документов в группах:
2      19
3       2
4    2387
Name: count, dtype: int64

Запросов без положительных примеров: 0
Запросов без отрицательных примеров: 0

Пример обучающей группы:


,id,relevance,selection_type
0,r_b5c61745207ac64d4964,1,positive
1,r_6986c9ce8924aa47363b,0,random_fill
2,r_5b8619602bc083089f02,1,positive
3,r_2a9132d50090da1fccbf,0,random_negative


In [ ]:

import torch.nn.functional as F

SCORE_SCALE = 8.0


def listwise_ranking_loss(
    scores,
    relevance,
    score_scale=SCORE_SCALE
):
    # Переводим оценки в FP32 для устойчивости
    logits = scores.float().reshape(-1) * score_scale
    labels = relevance.float().reshape(-1)

    if logits.numel() != labels.numel():
        raise ValueError(
            "Количество оценок и меток не совпадает"
        )

    if torch.any(labels < 0):
        raise ValueError(
            "Метки релевантности должны быть неотрицательными"
        )

    if labels.sum().item() <= 0:
        raise ValueError(
            "Группа не содержит положительных примеров"
        )

    # Целевое распределение по релевантным документам
    target = labels / labels.sum()

    # Listwise cross-entropy
    log_probs = F.log_softmax(logits, dim=0)

    loss = -(target * log_probs).sum()

    return loss


# Проверяем функцию на искусственном примере
test_scores = torch.tensor(
    [0.25, -0.10, 0.80, -0.40],
    dtype=torch.float32,
    requires_grad=True
)

test_labels = torch.tensor(
    [1.0, 0.0, 1.0, 0.0],
    dtype=torch.float32
)

test_loss = listwise_ranking_loss(
    test_scores,
    test_labels
)

test_loss.backward()

assert torch.isfinite(test_loss).item()
assert test_scores.grad is not None
assert torch.isfinite(test_scores.grad).all().item()

print("Score scale:", SCORE_SCALE)
print("Listwise Loss:", round(test_loss.item(), 6))
print("Градиенты:", test_scores.grad.tolist())
print("\nListwise Ranking Loss работает корректно!")


Score scale: 8.0
Listwise Loss: 2.213007
Градиенты: [-3.903050422668457, 0.005895502865314484, 3.8966197967529297, 0.0005348276463337243]

Listwise Ranking Loss работает корректно!


In [ ]:

# Включаем режим обучения
model.train()

model.gradient_checkpointing_enable(
    gradient_checkpointing_kwargs={
        "use_reentrant": False
    }
)
model.enable_input_require_grads()

# Получаем исходную модель с подключёнными LoRA-слоями
base_ranker = model.get_base_model()


def prepare_training_documents(group):
    documents = []

    for document in group["document"].fillna("").astype(str):
        token_ids = tokenizer.encode(
            document,
            add_special_tokens=False,
            truncation=True,
            max_length=MAX_DOC_TOKENS
        )

        documents.append(
            tokenizer.decode(
                token_ids,
                skip_special_tokens=True
            )
        )

    return documents


# Проверяем на одной обучающей группе
sample_group = training_groups[0]

query = str(sample_group["query"].iloc[0])
documents = prepare_training_documents(sample_group)

labels = torch.tensor(
    sample_group["relevance"].to_numpy(),
    dtype=torch.float32,
    device=DEVICE
)

model.zero_grad(set_to_none=True)

# Прямой проход с сохранением градиентов
outputs = base_ranker._compute_single_batch(
    query,
    documents
)

scores = outputs.scores.reshape(-1)

assert len(scores) == len(labels)
assert scores.requires_grad
assert torch.isfinite(scores).all()

# Расчёт функции потерь
loss = listwise_ranking_loss(scores, labels)

# Обратное распространение ошибки
loss.backward()

# Проверяем градиенты LoRA
grad_count = 0
nonzero_grad_count = 0

for name, param in model.named_parameters():
    if param.requires_grad and param.grad is not None:
        grad_count += 1

        if torch.isfinite(param.grad).all() and \
           torch.any(param.grad != 0):
            nonzero_grad_count += 1

assert nonzero_grad_count > 0, (
    "Градиенты не дошли до LoRA"
)

print("Запрос:", query)
print("Документов:", len(documents))
print("Оценки Jina:", scores.detach().float().cpu().tolist())
print("Метки:", labels.cpu().tolist())
print("Listwise Loss:", round(loss.item(), 6))

print("\nLoRA-тензоров с градиентами:", grad_count)
print("С ненулевыми градиентами:", nonzero_grad_count)

if DEVICE.type == "cuda":
    print(
        "Пиковая GPU-память:",
        round(torch.cuda.max_memory_allocated() / 1024**3, 2),
        "GB"
    )

# Очищаем память перед обучением
model.zero_grad(set_to_none=True)

del outputs, scores, loss
gc.collect()

if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

print("\nПроверка градиентов Jina + LoRA завершена!")


Запрос: Иоа́нн Тобо́льский был поэтом?
Документов: 4
Оценки Jina: [0.4365234375, 0.022430419921875, 0.2191162109375, -0.135498046875]
Метки: [1.0, 0.0, 1.0, 0.0]
Listwise Loss: 1.070413

LoRA-тензоров с градиентами: 112
С ненулевыми градиентами: 56
Пиковая GPU-память: 2.57 GB

Проверка градиентов Jina + LoRA завершена!


In [ ]:

from torch.optim import AdamW

# Гиперпараметры обучения
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
NUM_EPOCHS = 1
MAX_GRAD_NORM = 1.0
CHECKPOINT_EVERY = 200

# Папки для сохранения результатов
CHECKPOINT_DIR = OUTPUT_DIR / "lora_checkpoints"
FINAL_ADAPTER_DIR = OUTPUT_DIR / "lora_adapter_final"

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# Оптимизируем только обучаемые параметры LoRA
trainable_parameters = [
    param
    for param in model.parameters()
    if param.requires_grad
]

optimizer = AdamW(
    trainable_parameters,
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

# Масштабирование градиентов для FP16
scaler = torch.amp.GradScaler(
    "cuda",
    enabled=(DEVICE.type == "cuda")
)

# Воспроизводимый порядок обучающих запросов
training_order = np.random.default_rng(
    SEED
).permutation(len(training_groups)).tolist()

TOTAL_STEPS = len(training_order) * NUM_EPOCHS

assert TOTAL_STEPS == 2408
assert sum(
    p.numel() for p in trainable_parameters
) == 1_146_880

print("Конфигурация обучения")
print("Optimizer: AdamW")
print("Learning rate:", LEARNING_RATE)
print("Weight decay:", WEIGHT_DECAY)
print("Epochs:", NUM_EPOCHS)
print("Total steps:", TOTAL_STEPS)
print("Gradient clipping:", MAX_GRAD_NORM)
print("Mixed precision: FP16")
print("Gradient scaler enabled:", scaler.is_enabled())
print("Checkpoint every:", CHECKPOINT_EVERY, "steps")
print("Checkpoint directory:", CHECKPOINT_DIR)
print("Final adapter directory:", FINAL_ADAPTER_DIR)


Конфигурация обучения
Optimizer: AdamW
Learning rate: 2e-05
Weight decay: 0.01
Epochs: 1
Total steps: 2408
Gradient clipping: 1.0
Mixed precision: FP16
Gradient scaler enabled: True
Checkpoint every: 200 steps
Checkpoint directory: /content/results/lora_checkpoints
Final adapter directory: /content/results/lora_adapter_final


In [ ]:

import hashlib
import shutil

# 1. Постоянное хранение контрольных точек
try:
    from google.colab import drive

    if not Path("/content/drive/MyDrive").is_dir():
        drive.mount("/content/drive")

    RUN_DIR = Path(
        "/content/drive/MyDrive/AMML_Ranking_2026"
    )

except ImportError:
    RUN_DIR = OUTPUT_DIR

CHECKPOINT_DIR = RUN_DIR / "lora_checkpoints"
FINAL_ADAPTER_DIR = RUN_DIR / "lora_adapter_final"

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# Сохраняем промежуточные CSV
BACKUP_DIR = RUN_DIR / "intermediate"
BACKUP_DIR.mkdir(parents=True, exist_ok=True)

for source in [
    SCORES_PATH,
    REVERSE_PATH,
    COMBINED_PATH,
    MINED_PATH
]:
    if source.exists():
        shutil.copy2(
            source,
            BACKUP_DIR / source.name
        )

# Контроль идентичности обучающей выборки
DATA_FINGERPRINT = hashlib.sha256(
    training_data[
        ["id", "query_id", "relevance"]
    ].to_csv(index=False).encode("utf-8")
).hexdigest()

# 2. Сохранение контрольной точки
def save_training_checkpoint(step, loss_history, skipped):
    checkpoint_path = (
        CHECKPOINT_DIR / f"step_{step:06d}"
    )
    checkpoint_path.mkdir(parents=True, exist_ok=True)

    model.save_pretrained(
        checkpoint_path,
        safe_serialization=True
    )

    lora_weights = {
        name: param.detach().cpu().clone()
        for name, param in model.named_parameters()
        if param.requires_grad
    }

    state = {
        "step": step,
        "lora_weights": lora_weights,
        "optimizer": optimizer.state_dict(),
        "scaler": scaler.state_dict(),
        "training_order": training_order,
        "loss_history": loss_history.copy(),
        "skipped_steps": skipped,
        "data_fingerprint": DATA_FINGERPRINT,
        "torch_rng": torch.get_rng_state(),
        "cuda_rng": (
            torch.cuda.get_rng_state_all()
            if DEVICE.type == "cuda" else []
        )
    }

    temp_path = checkpoint_path / "training_state.tmp"
    final_path = checkpoint_path / "training_state.pt"

    torch.save(state, temp_path)
    temp_path.replace(final_path)


# 3. Восстановление после прерывания
start_step = 0
loss_history = []
skipped_steps = 0

saved_checkpoints = sorted(
    CHECKPOINT_DIR.glob("step_*/training_state.pt")
)

if saved_checkpoints:
    latest = saved_checkpoints[-1]

    state = torch.load(
        latest,
        map_location="cpu",
        weights_only=True
    )

    assert (
        state["data_fingerprint"] == DATA_FINGERPRINT
    ), "Обучающая выборка изменилась!"

    assert (
        state["training_order"] == training_order
    ), "Порядок обучения изменился!"

    named_parameters = dict(model.named_parameters())

    with torch.no_grad():
        for name, weights in state["lora_weights"].items():
            param = named_parameters[name]
            param.copy_(
                weights.to(
                    device=param.device,
                    dtype=param.dtype
                )
            )

    optimizer.load_state_dict(state["optimizer"])
    scaler.load_state_dict(state["scaler"])

    torch.set_rng_state(state["torch_rng"])

    if DEVICE.type == "cuda":
        torch.cuda.set_rng_state_all(state["cuda_rng"])

    start_step = int(state["step"])
    loss_history = list(state["loss_history"])
    skipped_steps = int(state["skipped_steps"])

    print("Восстановлен checkpoint:", latest.parent.name)
else:
    print("Начинаем обучение с первого шага")


# 4. Цикл обучения
model.train()
base_ranker = model.get_base_model()

progress = tqdm(
    range(start_step, TOTAL_STEPS),
    initial=start_step,
    total=TOTAL_STEPS,
    desc="LoRA training"
)

for position in progress:
    step = position + 1
    group = training_groups[training_order[position]]

    query = str(group["query"].iloc[0])
    documents = prepare_training_documents(group)

    labels = torch.tensor(
        group["relevance"].to_numpy(),
        dtype=torch.float32,
        device=DEVICE
    )

    optimizer.zero_grad(set_to_none=True)

    outputs = base_ranker._compute_single_batch(
        query,
        documents
    )

    scores = outputs.scores.reshape(-1)
    loss = listwise_ranking_loss(scores, labels)

    if not torch.isfinite(loss).item():
        raise RuntimeError(
            f"Некорректный Loss на шаге {step}"
        )

    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)

    torch.nn.utils.clip_grad_norm_(
        trainable_parameters,
        MAX_GRAD_NORM
    )

    previous_scale = scaler.get_scale()

    scaler.step(optimizer)
    scaler.update()

    if scaler.get_scale() < previous_scale:
        skipped_steps += 1

    loss_history.append(float(loss.item()))

    progress.set_postfix(
        loss=f"{loss.item():.4f}",
        avg50=f"{np.mean(loss_history[-50:]):.4f}"
    )

    del outputs, scores, loss

    if (
        step % CHECKPOINT_EVERY == 0
        or step == TOTAL_STEPS
    ):
        save_training_checkpoint(
            step,
            loss_history,
            skipped_steps
        )

        print(
            f"\nCheckpoint {step}/{TOTAL_STEPS} сохранён"
        )


# 5. Сохранение итогового адаптера
model.save_pretrained(
    FINAL_ADAPTER_DIR,
    safe_serialization=True
)

pd.DataFrame({
    "step": np.arange(1, len(loss_history) + 1),
    "loss": loss_history
}).to_csv(
    RUN_DIR / "training_history.csv",
    index=False
)

print("\nОбучение завершено!")
print("Всего шагов:", TOTAL_STEPS)
print("Пропущенных обновлений FP16:", skipped_steps)
print("Начальный Loss:", loss_history[0])
print("Средний Loss последних 50 шагов:",
      np.mean(loss_history[-50:]))
print("Адаптер сохранён:", FINAL_ADAPTER_DIR)


Mounted at /content/drive
Начинаем обучение с первого шага


LoRA training:   0%|          | 0/2408 [00:00<?, ?it/s]


Checkpoint 200/2408 сохранён

Checkpoint 400/2408 сохранён

Checkpoint 600/2408 сохранён

Checkpoint 800/2408 сохранён

Checkpoint 1000/2408 сохранён

Checkpoint 1200/2408 сохранён

Checkpoint 1400/2408 сохранён

Checkpoint 1600/2408 сохранён

Checkpoint 1800/2408 сохранён

Checkpoint 2000/2408 сохранён

Checkpoint 2200/2408 сохранён

Checkpoint 2400/2408 сохранён

Checkpoint 2408/2408 сохранён

Обучение завершено!
Всего шагов: 2408
Пропущенных обновлений FP16: 5
Начальный Loss: 1.0559337139129639
Средний Loss последних 50 шагов: 0.832227937139105
Адаптер сохранён: /content/drive/MyDrive/AMML_Ranking_2026/lora_adapter_final



## 8. Оценка качества модели

После дообучения Jina Reranker v3.5 с использованием LoRA качество ранжирования оценивается на двух отложенных выборках:

- **Validation** – 329 поисковых запросов.
- **Dev** – 305 поисковых запросов.

Разделение выполнено по `query_id`, поэтому запросы из оценочных выборок не использовались при обучении модели.

### Метод инференса

Для получения предсказаний используется двунаправленное ранжирование (Bidirectional Reranking):

1. Модель оценивает документы в исходном порядке.
2. Выполняется повторное ранжирование документов в обратном порядке.
3. Оценки каждого прохода стандартизируются методом Z-score внутри запроса.
4. Итоговая оценка рассчитывается как среднее стандартизированных оценок двух проходов.

При инференсе используются подготовленные тексты документов длиной до 1800 символов без дополнительной ручной обрезки до 192 токенов.

### Метрика качества

Основная метрика соревнования – **NDCG@10 (Normalized Discounted Cumulative Gain at 10)**.

Она оценивает качество ранжирования с учётом релевантности документов и их положения в списке. Чем выше расположен релевантный документ, тем больший вклад он вносит в итоговую метрику.

Для каждого запроса рассчитывается NDCG@10, после чего результаты усредняются по всем запросам выборки.

### Результаты

| Метод | Validation NDCG@10 | Dev NDCG@10 |
|---|---:|---:|
| Bidirectional LoRA | **0.969755** | **0.968230** |

Полученные результаты подтверждают высокое качество ранжирования на двух отложенных выборках.

Двунаправленный метод с усреднением оценок выбран для формирования итоговых предсказаний тестовой выборки.


In [33]:

# Вспомогательные функции для оценки качества

model.eval()
core = model.get_base_model()


def historical_rerank(query, documents):
    """Оценки Jina + LoRA с сохранением исходного порядка."""

    with torch.inference_mode():
        ranked = core.rerank(
            query,
            documents,
            top_n=None
        )

    scores = np.full(
        len(documents),
        np.nan,
        dtype=np.float64
    )

    for item in ranked:
        scores[int(item["index"])] = float(
            item["relevance_score"]
        )

    assert np.isfinite(scores).all()

    return scores


def normalize_group_scores(values):
    """Z-score нормализация внутри одного запроса."""

    values = np.asarray(values, dtype=np.float64)

    if len(values) < 2:
        return np.zeros_like(values)

    std = values.std(ddof=1)

    if not np.isfinite(std) or std < 1e-12:
        return np.zeros_like(values)

    return (values - values.mean()) / (std + 1e-6)


def calculate_ndcg_at_10(group, score_column):
    """Расчёт NDCG@10 для одного запроса."""

    relevance = group["relevance"].to_numpy(
        dtype=np.float64
    )

    scores = group[score_column].to_numpy(
        dtype=np.float64
    )

    k = min(10, len(group))

    ranking = np.argsort(
        -scores,
        kind="stable"
    )[:k]

    ranked_relevance = relevance[ranking]
    ideal_relevance = np.sort(relevance)[::-1][:k]

    discounts = np.log2(
        np.arange(2, k + 2)
    )

    dcg = np.sum(
        (2 ** ranked_relevance - 1) / discounts
    )

    idcg = np.sum(
        (2 ** ideal_relevance - 1) / discounts
    )

    return float(dcg / idcg) if idcg > 0 else 0.0


# Быстрая проверка функций на одном запросе
sample_group = next(iter(
    val.groupby("query_id", sort=True)
))[1].copy()

query = str(sample_group["query"].iloc[0])
documents = sample_group["document"].fillna(
    ""
).astype(str).tolist()

forward = historical_rerank(query, documents)

reverse = historical_rerank(
    query,
    documents[::-1]
)[::-1]

sample_group["score_combined"] = (
    normalize_group_scores(forward)
    + normalize_group_scores(reverse)
) / 2

sample_ndcg = calculate_ndcg_at_10(
    sample_group,
    "score_combined"
)

print("Функции инициализированы успешно!")
print("Проверочный запрос:", query)
print("Документов:", len(documents))
print("NDCG@10:", round(sample_ndcg, 6))
print("Все оценки конечны:", np.isfinite(
    sample_group["score_combined"]
).all())


Функции инициализированы успешно!
Проверочный запрос: Какая протяженность реки Колумбия?
Документов: 2
NDCG@10: 1.0
Все оценки конечны: True


In [34]:

# Оценка качества на Validation (329 запросов)

model.eval()

VAL_PRED_PATH = RUN_DIR / "validation_predictions_final.csv"
VAL_METRICS_PATH = RUN_DIR / "validation_metrics_final.csv"

validation_parts = []

for qid, group in tqdm(
    val.groupby("query_id", sort=True),
    total=len(val_qids),
    desc="Validation"
):
    query = str(group["query"].iloc[0])

    # Используем подготовленные документы без
    # дополнительной обрезки до 192 токенов
    documents = (
        group["document"]
        .fillna("")
        .astype(str)
        .tolist()
    )

    # Прямой и обратный проходы
    forward = historical_rerank(query, documents)
    reverse = historical_rerank(
        query, documents[::-1]
    )[::-1]

    # Z-score и усреднение
    combined = (
        normalize_group_scores(forward)
        + normalize_group_scores(reverse)
    ) / 2

    result = group[
        ["id", "query_id", "relevance"]
    ].copy()

    result["score_forward"] = forward
    result["score_reverse"] = reverse
    result["score_combined"] = combined

    validation_parts.append(result)

validation_predictions = pd.concat(
    validation_parts,
    ignore_index=True
)

# Проверки целостности
assert len(validation_predictions) == len(val)
assert validation_predictions["id"].is_unique
assert validation_predictions["query_id"].nunique() == 329

assert np.isfinite(
    validation_predictions[
        ["score_forward", "score_reverse", "score_combined"]
    ].to_numpy(dtype=float)
).all()

# Расчёт NDCG@10
validation_metrics = {}

for column in [
    "score_forward",
    "score_reverse",
    "score_combined"
]:
    validation_metrics[column] = float(np.mean([
        calculate_ndcg_at_10(group, column)
        for _, group in validation_predictions.groupby(
            "query_id", sort=True
        )
    ]))

metrics_df = pd.DataFrame([
    {"method": method, "ndcg_at_10": score}
    for method, score in validation_metrics.items()
])

# Сохраняем результаты
validation_predictions.to_csv(
    VAL_PRED_PATH, index=False
)

metrics_df.to_csv(
    VAL_METRICS_PATH, index=False
)

print("\nValidation завершена!")
print("Запросов:", validation_predictions["query_id"].nunique())
print("Документов:", len(validation_predictions))

print("\nNDCG@10:")
display(
    metrics_df.sort_values(
        "ndcg_at_10",
        ascending=False
    )
)

print(
    "\nИтоговый Validation NDCG@10:",
    round(validation_metrics["score_combined"], 6)
)

print("\nРезультаты сохранены:")
print(VAL_PRED_PATH)
print(VAL_METRICS_PATH)


Validation:   0%|          | 0/329 [00:00<?, ?it/s]


Validation завершена!
Запросов: 329
Документов: 3426

NDCG@10:


,method,ndcg_at_10
1,score_reverse,0.970361
2,score_combined,0.969755
0,score_forward,0.964010



Итоговый Validation NDCG@10: 0.969755

Результаты сохранены:
/content/drive/MyDrive/AMML_Ranking_2026/validation_predictions_final.csv
/content/drive/MyDrive/AMML_Ranking_2026/validation_metrics_final.csv


In [35]:

# Независимая оценка качества на Dev (305 запросов)

model.eval()

DEV_PRED_PATH = RUN_DIR / "dev_predictions_final.csv"
DEV_METRICS_PATH = RUN_DIR / "dev_metrics_final.csv"

dev_parts = []

for qid, group in tqdm(
    dev.groupby("query_id", sort=True),
    total=len(dev_qids),
    desc="Dev"
):
    query = str(group["query"].iloc[0])

    # Подготовленные тексты без дополнительной
    # обрезки до 192 токенов
    documents = (
        group["document"]
        .fillna("")
        .astype(str)
        .tolist()
    )

    # Прямой и обратный проходы
    forward = historical_rerank(query, documents)

    reverse = historical_rerank(
        query, documents[::-1]
    )[::-1]

    # Z-score и усреднение
    combined = (
        normalize_group_scores(forward)
        + normalize_group_scores(reverse)
    ) / 2

    result = group[
        ["id", "query_id", "relevance"]
    ].copy()

    result["score_forward"] = forward
    result["score_reverse"] = reverse
    result["score_combined"] = combined

    dev_parts.append(result)

dev_predictions = pd.concat(
    dev_parts,
    ignore_index=True
)

# Проверки целостности
assert len(dev_predictions) == len(dev)
assert dev_predictions["id"].is_unique
assert set(dev_predictions["id"]) == set(dev["id"])
assert dev_predictions["query_id"].nunique() == 305

assert np.isfinite(
    dev_predictions[
        ["score_forward", "score_reverse", "score_combined"]
    ].to_numpy(dtype=float)
).all()

# Расчёт NDCG@10
dev_metrics = {}

for column in [
    "score_forward",
    "score_reverse",
    "score_combined"
]:
    dev_metrics[column] = float(np.mean([
        calculate_ndcg_at_10(group, column)
        for _, group in dev_predictions.groupby(
            "query_id", sort=True
        )
    ]))

dev_metrics_df = pd.DataFrame([
    {"method": method, "ndcg_at_10": score}
    for method, score in dev_metrics.items()
])

# Сохранение результатов
dev_predictions.to_csv(
    DEV_PRED_PATH, index=False
)

dev_metrics_df.to_csv(
    DEV_METRICS_PATH, index=False
)

print("\nDev завершена!")
print("Запросов:", dev_predictions["query_id"].nunique())
print("Документов:", len(dev_predictions))

print("\nNDCG@10:")
display(
    dev_metrics_df.sort_values(
        "ndcg_at_10",
        ascending=False
    )
)

print(
    "\nИтоговый Dev NDCG@10:",
    round(dev_metrics["score_combined"], 6)
)

print("\nРезультаты сохранены:")
print(DEV_PRED_PATH)
print(DEV_METRICS_PATH)


Dev:   0%|          | 0/305 [00:00<?, ?it/s]


Dev завершена!
Запросов: 305
Документов: 3160

NDCG@10:


,method,ndcg_at_10
2,score_combined,0.968230
1,score_reverse,0.966902
0,score_forward,0.964440



Итоговый Dev NDCG@10: 0.96823

Результаты сохранены:
/content/drive/MyDrive/AMML_Ranking_2026/dev_predictions_final.csv
/content/drive/MyDrive/AMML_Ranking_2026/dev_metrics_final.csv



## 9. Предсказания на тестовой выборке

После проверки качества на Validation и Dev дообученная модель Jina Reranker v3.5 с LoRA используется для ранжирования документов тестовой выборки.

Для получения итоговых оценок применяется выбранный по результатам валидации метод двунаправленного ранжирования:

1. Для каждого запроса рассчитываются оценки документов в исходном порядке.
2. Выполняется повторный проход с обратным порядком документов.
3. Оценки каждого прохода стандартизируются методом Z-score внутри запроса.
4. Итоговая оценка определяется как среднее стандартизированных оценок двух проходов.

При инференсе используются тексты длиной до 1800 символов без дополнительной ручной обрезки до 192 токенов. Это соответствует методу, показавшему лучшие результаты на отложенных выборках.

Перед ранжированием применяются ранее выполненные исправления повреждённой кодировки тестовых документов.

Итоговые предсказания сохраняются в `submission.csv` в формате, соответствующем `sample_submission.csv`.

Дополнительно сохраняются оценки прямого, обратного и объединённого проходов для анализа и воспроизводимости результата.


In [32]:

import hashlib

model.eval()
core = model.get_base_model()

TEST_PRED_PATH = RUN_DIR / "test_predictions.csv"
SUBMISSION_PATH = RUN_DIR / "submission.csv"

assert {"id", "query_id", "query", "document"}.issubset(test.columns)
assert test["id"].is_unique
assert sample["id"].is_unique
assert set(test["id"]) == set(sample["id"])

# Определяем формат submission до запуска расчётов
target_columns = [
    column for column in sample.columns
    if column not in ["id", "query_id"]
]

assert len(target_columns) == 1, (
    f"Неожиданный формат sample_submission: {sample.columns.tolist()}"
)

TARGET_COLUMN = target_columns[0]

if "rank" in TARGET_COLUMN.lower():
    raise ValueError(
        "Шаблон ожидает ранги, а не оценки. "
        "Необходимо проверить формат соревнования."
    )

print("Столбцы submission:", sample.columns.tolist())
print("Столбец предсказания:", TARGET_COLUMN)

# Уникальный идентификатор модели и тестовых данных
fingerprint = hashlib.sha256()

for name, param in model.named_parameters():
    if param.requires_grad:
        fingerprint.update(name.encode())
        fingerprint.update(
            param.detach().cpu().contiguous().numpy().tobytes()
        )

fingerprint.update(
    pd.util.hash_pandas_object(
        test[["id", "document"]],
        index=False
    ).values.tobytes()
)

RUN_HASH = fingerprint.hexdigest()[:12]
PARTIAL_PATH = RUN_DIR / f"test_partial_{RUN_HASH}.csv"

columns = [
    "id", "query_id",
    "score_forward", "score_reverse", "score_combined"
]

if PARTIAL_PATH.exists():
    saved = pd.read_csv(
        PARTIAL_PATH,
        dtype={"id": str, "query_id": str}
    )
    print("Найдены промежуточные результаты:", len(saved))
else:
    saved = pd.DataFrame(columns=columns)

assert saved["id"].is_unique
assert set(saved["id"]).issubset(set(test["id"]))

expected = test.groupby("query_id").size()
actual = saved.groupby("query_id").size()

completed_qids = {
    qid for qid, count in actual.items()
    if count == expected.loc[qid]
    and set(saved.loc[saved["query_id"] == qid, "id"])
    == set(test.loc[test["query_id"] == qid, "id"])
}

saved = saved[
    saved["query_id"].isin(completed_qids)
].copy()

pending = []

# Вычисление предсказаний
for qid, group in tqdm(
    test.groupby("query_id", sort=True),
    total=test["query_id"].nunique(),
    desc="Test inference"
):
    if qid in completed_qids:
        continue

    query = str(group["query"].iloc[0])

    # Исправленные тексты из раздела 2
    documents = group["document"].fillna("").astype(str).tolist()

    # Прямой проход
    forward = historical_rerank(
        query,
        documents
    )

    # Обратный проход
    reverse = historical_rerank(
        query,
        documents[::-1]
    )[::-1]

    combined = (
        normalize_group_scores(forward)
        + normalize_group_scores(reverse)
    ) / 2

    result = group[["id", "query_id"]].copy()
    result["score_forward"] = forward
    result["score_reverse"] = reverse
    result["score_combined"] = combined

    pending.append(result)

    if len(pending) >= 50:
        batch = pd.concat(pending, ignore_index=True)
        saved = pd.concat([saved, batch], ignore_index=True)

        temp_path = PARTIAL_PATH.with_suffix(".tmp")
        saved.to_csv(temp_path, index=False)
        temp_path.replace(PARTIAL_PATH)

        pending = []

if pending:
    saved = pd.concat(
        [saved, *pending],
        ignore_index=True
    )
    saved.to_csv(PARTIAL_PATH, index=False)

# Проверки
assert len(saved) == len(test)
assert saved["id"].is_unique
assert set(saved["id"]) == set(test["id"])
assert saved["query_id"].nunique() == test["query_id"].nunique()

assert np.isfinite(
    saved[
        ["score_forward", "score_reverse", "score_combined"]
    ].to_numpy(dtype=float)
).all()

# Сохраняем полные оценки
saved.to_csv(TEST_PRED_PATH, index=False)

# Формируем submission в порядке sample_submission.csv
score_map = saved.set_index("id")["score_combined"]

submission = sample.copy()
submission[TARGET_COLUMN] = submission["id"].map(score_map)

assert submission.columns.tolist() == sample.columns.tolist()
assert submission["id"].tolist() == sample["id"].tolist()
assert submission[TARGET_COLUMN].notna().all()
assert np.isfinite(
    submission[TARGET_COLUMN].to_numpy(dtype=float)
).all()

submission.to_csv(SUBMISSION_PATH, index=False)
submission.to_csv(DATA_DIR / "submission.csv", index=False)

print("\nTest inference завершён!")
print("Запросов:", saved["query_id"].nunique())
print("Документов:", len(saved))
print("Строк submission:", len(submission))

print("\nФайлы сохранены:")
print(TEST_PRED_PATH)
print(SUBMISSION_PATH)

print("\nПервые строки submission:")
display(submission.head())


Столбцы submission: ['id', 'prediction']
Столбец предсказания: prediction
Найдены промежуточные результаты: 13507


Test inference:   0%|          | 0/1305 [00:00<?, ?it/s]


Test inference завершён!
Запросов: 1305
Документов: 13507
Строк submission: 13507

Файлы сохранены:
/content/drive/MyDrive/AMML_Ranking_2026/test_predictions.csv
/content/drive/MyDrive/AMML_Ranking_2026/submission.csv

Первые строки submission:


,id,prediction
0,r_00100bd94cf8513c6500,0.431182
1,r_0011d1a81cda1b1c720f,-0.483524
2,r_00135550b108ebd88432,-0.423260
3,r_001cac3cb24c86e57ef0,-0.806122
4,r_001e8de6d4886f85b86e,-0.344235
